In [ ]:
!pip install ultralytics opencv-python-headless matplotlib numpy -q
print('✅ All dependencies installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.3 MB/s eta 0:00:00
✅ All dependencies installed!


In [ ]:
from google.colab import files

print('📤 Please upload your CCTV video file...')
uploaded = files.upload()

VIDEO_PATH = list(uploaded.keys())[0]
print(f'✅ Video uploaded: {VIDEO_PATH}')

📤 Please upload your CCTV video file...


Saving Video Project 1 (2).mp4 to Video Project 1 (2).mp4
✅ Video uploaded: Video Project 1 (2).mp4


In [ ]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Image as IPImage
import io
from PIL import Image

# ── Extract first frame ──────────────────────────────────────────────────────
cap = cv2.VideoCapture(VIDEO_PATH)
ret, first_frame = cap.read()
cap.release()

if not ret:
    raise ValueError('❌ Could not read video.')

first_frame_rgb = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)
H, W = first_frame.shape[:2]
print(f'✅ Frame: {W}x{H}')

# ── State ─────────────────────────────────────────────────────────────────────
tripwire_points = []
zone_points     = []
draw_mode       = {'current': None}  # 'tripwire' or 'zone'

# ── Convert frame to displayable bytes ───────────────────────────────────────
def frame_to_bytes(frame_rgb):
    annotated = frame_rgb.copy()

    # Draw tripwire
    if len(tripwire_points) == 2:
        p1 = tripwire_points[0]; p2 = tripwire_points[1]
        cv2.line(annotated, p1, p2, (255,0,0), 3)
        cv2.putText(annotated, 'TRIPWIRE', (p1[0], p1[1]-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,0,0), 2)
    for p in tripwire_points:
        cv2.circle(annotated, p, 7, (255,0,0), -1)

    # Draw zone
    if len(zone_points) >= 2:
        for i in range(len(zone_points)-1):
            cv2.line(annotated, zone_points[i], zone_points[i+1], (0,100,255), 2)
        if len(zone_points) >= 3:
            pts_arr = np.array(zone_points, dtype=np.int32).reshape((-1,1,2))
            overlay = annotated.copy()
            cv2.fillPoly(overlay, [pts_arr], (0,100,255))
            cv2.addWeighted(overlay, 0.2, annotated, 0.8, 0, annotated)
            cv2.polylines(annotated, [pts_arr], True, (0,100,255), 2)
            cx = int(np.mean([p[0] for p in zone_points]))
            cy = int(np.mean([p[1] for p in zone_points]))
            cv2.putText(annotated, 'LOITER ZONE', (cx-60, cy),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,100,255), 2)
    for p in zone_points:
        cv2.circle(annotated, p, 7, (0,100,255), -1)

    img_pil = Image.fromarray(annotated)
    buf = io.BytesIO()
    img_pil.save(buf, format='JPEG', quality=90)
    return buf.getvalue()

# ── Display image widget ──────────────────────────────────────────────────────
img_widget = widgets.Image(value=frame_to_bytes(first_frame_rgb),
                           format='jpeg', width=W if W<1000 else 900)

# ── Mode label ────────────────────────────────────────────────────────────────
status_label = widgets.Label(value='👆 Select a mode below, then enter coordinates')

# ── Coordinate input boxes ────────────────────────────────────────────────────
x_input = widgets.IntText(value=0, description='X:', layout=widgets.Layout(width='150px'))
y_input = widgets.IntText(value=0, description='Y:', layout=widgets.Layout(width='150px'))

# ── Buttons ───────────────────────────────────────────────────────────────────
btn_tw_mode  = widgets.Button(description='🔴 Tripwire Mode',
                               button_style='danger',
                               layout=widgets.Layout(width='160px'))
btn_zo_mode  = widgets.Button(description='🔵 Zone Mode',
                               button_style='info',
                               layout=widgets.Layout(width='160px'))
btn_add      = widgets.Button(description='➕ Add Point',
                               button_style='success',
                               layout=widgets.Layout(width='140px'))
btn_reset    = widgets.Button(description='🔄 Reset All',
                               button_style='warning',
                               layout=widgets.Layout(width='140px'))
btn_done     = widgets.Button(description='✅ Done / Close Zone',
                               button_style='primary',
                               layout=widgets.Layout(width='180px'))

def on_tw_mode(b):
    draw_mode['current'] = 'tripwire'
    tripwire_points.clear()
    status_label.value = '🔴 TRIPWIRE MODE — Enter X,Y and click ➕ Add Point (need 2 points)'
    img_widget.value = frame_to_bytes(first_frame_rgb)

def on_zo_mode(b):
    draw_mode['current'] = 'zone'
    zone_points.clear()
    status_label.value = '🔵 ZONE MODE — Enter X,Y and click ➕ Add Point (min 3 points, then ✅ Done)'
    img_widget.value = frame_to_bytes(first_frame_rgb)

def on_add(b):
    x = x_input.value
    y = y_input.value
    m = draw_mode['current']

    if m is None:
        status_label.value = '⚠️ Select a mode first!'
        return

    if m == 'tripwire':
        if len(tripwire_points) >= 2:
            status_label.value = '⚠️ Tripwire already has 2 points. Reset to redo.'
            return
        tripwire_points.append((x, y))
        status_label.value = f'🔴 Tripwire point {len(tripwire_points)}/2 added: ({x},{y})'
        if len(tripwire_points) == 2:
            status_label.value = f'✅ Tripwire done! {tripwire_points}'

    elif m == 'zone':
        if len(zone_points) >= 10:
            status_label.value = '⚠️ Maximum 10 points reached for zone!'
            return
        zone_points.append((x, y))
        remaining = 4 - len(zone_points)
        if remaining > 0:
            status_label.value = f'🔵 Zone point {len(zone_points)}/10 added: ({x},{y}) — need {remaining} more minimum'
        else:
            status_label.value = f'🔵 Zone point {len(zone_points)}/10 added: ({x},{y}) — click ✅ Done when finished (max 10)'

    img_widget.value = frame_to_bytes(first_frame_rgb)

def on_done(b):
    if draw_mode['current'] == 'zone':
        if len(zone_points) < 4:
            status_label.value = f'⚠️ Zone needs at least 4 points! You have {len(zone_points)} so far.'
            return
        draw_mode['current'] = None
        status_label.value = f'✅ Zone closed with {len(zone_points)} points: {zone_points}'
        img_widget.value = frame_to_bytes(first_frame_rgb)
    elif draw_mode['current'] == 'tripwire':
        if len(tripwire_points) < 2:
            status_label.value = f'⚠️ Tripwire needs exactly 2 points! You have {len(tripwire_points)}.'
            return
        draw_mode['current'] = None
        status_label.value = f'✅ Tripwire set: {tripwire_points}'
    else:
        status_label.value = '⚠️ Nothing to close. Select a mode first.'

def on_reset(b):
    tripwire_points.clear()
    zone_points.clear()
    draw_mode['current'] = None
    status_label.value = '🔄 Reset! Select a mode to start again.'
    img_widget.value = frame_to_bytes(first_frame_rgb)

btn_tw_mode.on_click(on_tw_mode)
btn_zo_mode.on_click(on_zo_mode)
btn_add.on_click(on_add)
btn_done.on_click(on_done)
btn_reset.on_click(on_reset)

# ── Layout ────────────────────────────────────────────────────────────────────
hint = widgets.HTML(value="""
<div style="background:#fff8e1;padding:10px;border-radius:6px;margin-bottom:8px;font-size:13px">
  <b>💡 How to find X, Y coordinates:</b><br>
  Hover your mouse over the image above → the coordinates show at the
  <b>bottom-right of your browser</b> or in the image toolbar.<br>
  Or: right-click the frame image → <i>Save image</i> → open in Paint/Preview → hover to get pixel position.
</div>
""")

controls = widgets.HBox([btn_tw_mode, btn_zo_mode,
                          x_input, y_input,
                          btn_add, btn_done, btn_reset])

display(img_widget, hint, status_label, controls)
print('\n✅ Widget ready! Use the buttons above to define your tripwire and zone.')

✅ Frame: 1280x720


Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x03\x02\x0…

HTML(value='\n<div style="background:#fff8e1;padding:10px;border-radius:6px;margin-bottom:8px;font-size:13px">…

Label(value='👆 Select a mode below, then enter coordinates')


✅ Widget ready! Use the buttons above to define your tripwire and zone.


In [ ]:
# # Verify your drawings
# print('📐 Your Configuration:')
# print(f'  Tripwire points : {tripwire_points}')
# print(f'  Zone points     : {zone_points}')

# if not tripwire_points and not zone_points:
#     print('\n⚠️  No tripwire or zone defined! Go back and draw at least one.')
# else:
#     print('\n✅ Configuration looks good!')

# # ── Rules — edit these values as needed ───────────────────────────────────────
# # Tripwire direction options: 'both' | 'left_to_right' | 'right_to_left'
# TRIPWIRE_DIRECTION = 'both'

# # How many seconds before loitering alert triggers
# LOITER_SECONDS = 3

# # YOLO minimum confidence (0.0 to 1.0)
# CONF_THRESHOLD = 0.4

# print(f'\n⚙️  Rules:')
# print(f'  Tripwire direction : {TRIPWIRE_DIRECTION}')
# print(f'  Loitering limit    : {LOITER_SECONDS} seconds')
# print(f'  YOLO confidence    : {CONF_THRESHOLD}')

In [ ]:
# Verify your drawings
print('📐 Your Configuration:')
print(f'  Tripwire points : {tripwire_points}')
print(f'  Zone points     : {zone_points}')

if not tripwire_points and not zone_points:
    print('\n⚠️  No tripwire or zone defined! Go back and draw at least one.')
else:
    print('\n✅ Configuration looks good!')

# ── Rules ─────────────────────────────────────────────────────────────────────
TRIPWIRE_DIRECTION      = 'both'   # 'both' | 'left_to_right' | 'right_to_left'
LOITER_SECONDS          = 5        # seconds before loitering alert triggers
CONF_THRESHOLD          = 0.3      # lowered slightly so more objects are caught
LOITER_DISPLAY_INTERVAL = 3        # seconds between loiter banner refreshes

# Detect ALL objects (None = all classes)
# Or restrict to specific classes, e.g. [0, 2, 16] = person, car, dog
DETECT_CLASSES = None

print(f'\n⚙️  Rules:')
print(f'  Tripwire direction      : {TRIPWIRE_DIRECTION}')
print(f'  Loitering limit         : {LOITER_SECONDS} seconds')
print(f'  YOLO confidence         : {CONF_THRESHOLD}')
print(f'  Detecting classes       : {"ALL objects" if DETECT_CLASSES is None else DETECT_CLASSES}')

📐 Your Configuration:
  Tripwire points : [(676, 168), (1188, 254)]
  Zone points     : [(324, 423), (968, 576), (1156, 320), (624, 198), (468, 547)]

✅ Configuration looks good!

⚙️  Rules:
  Tripwire direction      : both
  Loitering limit         : 5 seconds
  YOLO confidence         : 0.3
  Detecting classes       : ALL objects


In [ ]:
# from ultralytics import YOLO
# import cv2
# import numpy as np

# # ── Load YOLOv8 model ─────────────────────────────────────────────────────────
# print('🔄 Loading YOLOv8 model...')
# model = YOLO('yolov8n.pt')
# print('✅ Model loaded!')

# # ── Helper functions ──────────────────────────────────────────────────────────
# def side_of_line(p, a, b):
#     return (b[0]-a[0])*(p[1]-a[1]) - (b[1]-a[1])*(p[0]-a[0])

# def check_tripwire_cross(prev_center, curr_center, p1, p2, direction):
#     if prev_center is None:
#         return False
#     prev_side = side_of_line(prev_center, p1, p2)
#     curr_side = side_of_line(curr_center, p1, p2)
#     if prev_side * curr_side >= 0:
#         return False
#     if direction == 'both':
#         return True
#     elif direction == 'left_to_right':
#         return prev_side > 0
#     elif direction == 'right_to_left':
#         return prev_side < 0
#     return False

# def point_in_polygon(point, polygon):
#     if len(polygon) < 3:
#         return False
#     x, y = point
#     inside = False
#     px, py = polygon[-1]
#     for cx, cy in polygon:
#         if ((cy > y) != (py > y)) and (x < (px-cx)*(y-cy)/(py-cy)+cx):
#             inside = not inside
#         px, py = cx, cy
#     return inside

# def is_near_tripwire(center, p1, p2, threshold=40):
#     """Check if person center is within threshold pixels of the tripwire line."""
#     x0,y0 = center
#     x1,y1 = p1
#     x2,y2 = p2
#     dx, dy = x2-x1, y2-y1
#     length = max(np.sqrt(dx*dx + dy*dy), 1)
#     # Perpendicular distance from point to line segment
#     dist = abs(dy*x0 - dx*y0 + x2*y1 - y2*x1) / length
#     # Also check if projection falls within segment
#     t = ((x0-x1)*dx + (y0-y1)*dy) / (length*length)
#     return dist < threshold and 0 <= t <= 1

# def draw_alert_banner(frame, text, color_bgr, alpha=0.6):
#     """Draw a semi-transparent alert banner at the top of the frame."""
#     H, W = frame.shape[:2]
#     banner_h = 60
#     overlay = frame.copy()
#     cv2.rectangle(overlay, (0, 0), (W, banner_h), color_bgr, -1)
#     cv2.addWeighted(overlay, alpha, frame, 1-alpha, 0, frame)
#     # Centered text
#     font = cv2.FONT_HERSHEY_DUPLEX
#     font_scale = 1.1
#     thickness = 2
#     text_size = cv2.getTextSize(text, font, font_scale, thickness)[0]
#     text_x = (W - text_size[0]) // 2
#     text_y = (banner_h + text_size[1]) // 2
#     cv2.putText(frame, text, (text_x, text_y), font, font_scale, (255,255,255), thickness)

# # ── Video setup ───────────────────────────────────────────────────────────────
# cap   = cv2.VideoCapture(VIDEO_PATH)
# FPS   = cap.get(cv2.CAP_PROP_FPS) or 25
# TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
# W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# OUTPUT_PATH = 'output_detection.mp4'
# fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# out    = cv2.VideoWriter(OUTPUT_PATH, fourcc, FPS, (W, H))

# print(f'📹 Video: {W}x{H} @ {FPS:.1f} FPS | {TOTAL} frames')

# # ── Tracking state ────────────────────────────────────────────────────────────
# prev_centers      = {}   # track_id → last center
# zone_entry_time   = {}   # track_id → timestamp when entered loiter zone
# crossed_ids       = set() # track_ids that crossed tripwire
# tripwire_alert_ids = set() # track_ids currently near tripwire (for sustained alert)
# alert_log         = []

# # ── Loiter alert display cooldown (show alert every N seconds on screen) ──────
# LOITER_DISPLAY_INTERVAL = 3   # seconds between on-screen loiter alert refreshes
# last_loiter_display     = {}  # track_id → last time loiter banner was shown

# frame_idx = 0
# print('\n🚀 Running detection...\n')

# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     timestamp = frame_idx / FPS
#     frame_idx += 1

#     # YOLO tracking
#     results = model.track(frame, persist=True, classes=[0],
#                           conf=CONF_THRESHOLD, verbose=False)

#     annotated = frame.copy()

#     # ── Draw tripwire ─────────────────────────────────────────────────────────
#     if len(tripwire_points) == 2:
#         # Color turns red if anyone is currently in tripwire zone
#         tw_color = (0,0,255) if tripwire_alert_ids else (0,200,255)
#         cv2.line(annotated, tripwire_points[0], tripwire_points[1], tw_color, 3)
#         cv2.putText(annotated, 'TRIPWIRE',
#                     (tripwire_points[0][0], tripwire_points[0][1]-12),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.7, tw_color, 2)

#     # ── Draw zone ─────────────────────────────────────────────────────────────
#     if len(zone_points) >= 3:
#         pts_arr = np.array(zone_points, dtype=np.int32).reshape((-1,1,2))
#         overlay = annotated.copy()
#         cv2.fillPoly(overlay, [pts_arr], (255,100,0))
#         cv2.addWeighted(overlay, 0.2, annotated, 0.8, 0, annotated)
#         cv2.polylines(annotated, [pts_arr], True, (255,100,0), 2)
#         cx = int(np.mean([p[0] for p in zone_points]))
#         cy = int(np.mean([p[1] for p in zone_points]))
#         cv2.putText(annotated, 'LOITER ZONE', (cx-60, cy),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,140,0), 2)

#     # ── Per-frame alert flags ─────────────────────────────────────────────────
#     active_tripwire_alert  = False
#     active_loiter_alert    = False
#     active_loiter_ids      = []
#     tripwire_alert_ids_this_frame = set()

#     # ── Process detections ────────────────────────────────────────────────────
#     if results[0].boxes is not None and results[0].boxes.id is not None:
#         boxes     = results[0].boxes.xyxy.cpu().numpy().astype(int)
#         track_ids = results[0].boxes.id.cpu().numpy().astype(int)
#         confs     = results[0].boxes.conf.cpu().numpy()

#         for box, tid, conf in zip(boxes, track_ids, confs):
#             x1,y1,x2,y2 = box
#             cx_p = (x1+x2)//2
#             cy_p = (y1+y2)//2
#             center = (cx_p, cy_p)
#             color = (0,220,0)
#             label_lines = [f'ID:{tid}  {conf:.2f}']

#             # ── Tripwire: detect crossing ──────────────────────────────────
#             if len(tripwire_points) == 2:
#                 crossed_now = check_tripwire_cross(
#                     prev_centers.get(tid), center,
#                     tripwire_points[0], tripwire_points[1],
#                     TRIPWIRE_DIRECTION)

#                 if crossed_now and tid not in crossed_ids:
#                     crossed_ids.add(tid)
#                     alert_log.append({
#                         'type'      : 'Tripwire',
#                         'track_id'  : int(tid),
#                         'timestamp' : f'{timestamp:.2f}s',
#                         'frame'     : frame_idx,
#                         'location'  : center
#                     })
#                     print(f'  🚨 TRIPWIRE CROSSED | ID:{tid} | {timestamp:.2f}s')

#                 # ── Sustained alert: person still near tripwire line ───────
#                 if is_near_tripwire(center, tripwire_points[0], tripwire_points[1], threshold=50):
#                     tripwire_alert_ids_this_frame.add(tid)
#                     active_tripwire_alert = True
#                     color = (0,0,255)
#                     label_lines.append('⚡ TRIPWIRE!')

#             # ── Loitering check ────────────────────────────────────────────
#             if len(zone_points) >= 3 and point_in_polygon(center, zone_points):
#                 if tid not in zone_entry_time:
#                     zone_entry_time[tid] = timestamp
#                 dwell = timestamp - zone_entry_time[tid]

#                 if dwell >= LOITER_SECONDS:
#                     active_loiter_alert = True
#                     active_loiter_ids.append((tid, dwell))
#                     color = (0,100,255)
#                     label_lines.append(f'⏱ {dwell:.0f}s')

#                     # Log to alert_log every 2 seconds per person
#                     last_log = next((a for a in reversed(alert_log)
#                                      if a['type']=='Loitering'
#                                      and a['track_id']==int(tid)), None)
#                     if (last_log is None or
#                             timestamp - float(last_log['timestamp'].replace('s','')) >= 2):
#                         alert_log.append({
#                             'type'      : 'Loitering',
#                             'track_id'  : int(tid),
#                             'timestamp' : f'{timestamp:.2f}s',
#                             'frame'     : frame_idx,
#                             'dwell_sec' : f'{dwell:.1f}'
#                         })
#             else:
#                 zone_entry_time.pop(tid, None)

#             # ── Draw bounding box ──────────────────────────────────────────
#             cv2.rectangle(annotated, (x1,y1), (x2,y2), color, 2)
#             cv2.circle(annotated, center, 5, color, -1)
#             for i, lbl in enumerate(label_lines):
#                 cv2.putText(annotated, lbl, (x1, y1 - 12 - i*18),
#                             cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

#             prev_centers[tid] = center

#     # Update sustained tripwire alert set
#     tripwire_alert_ids = tripwire_alert_ids_this_frame

#     # ── ON-SCREEN ALERT BANNERS ───────────────────────────────────────────────

#     # Tripwire banner — shows every frame while person is near line
#     if active_tripwire_alert:
#         draw_alert_banner(annotated,
#                           '🚨  ALERT: TRIPWIRE BREACH DETECTED!',
#                           (0, 0, 180), alpha=0.65)

#     # Loitering banner — shows every LOITER_DISPLAY_INTERVAL seconds
#     elif active_loiter_alert:
#         # Pick the person loitering longest
#         tid_max, dwell_max = max(active_loiter_ids, key=lambda x: x[1])
#         last_shown = last_loiter_display.get(tid_max, -999)
#         if timestamp - last_shown >= LOITER_DISPLAY_INTERVAL:
#             last_loiter_display[tid_max] = timestamp
#         # Show banner only within 1.5s of last display trigger
#         if timestamp - last_loiter_display.get(tid_max, -999) < 1.5:
#             draw_alert_banner(annotated,
#                               f'⚠️  ALERT: LOITERING DETECTED  ({dwell_max:.0f}s)',
#                               (0, 80, 200), alpha=0.60)

#     # ── Timestamp ─────────────────────────────────────────────────────────────
#     cv2.putText(annotated, f'Frame:{frame_idx}  Time:{timestamp:.2f}s',
#                 (10, H-15), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)

#     out.write(annotated)

#     if frame_idx % 50 == 0:
#         pct = frame_idx/TOTAL*100 if TOTAL > 0 else 0
#         print(f'  Progress: {frame_idx}/{TOTAL} ({pct:.1f}%)')

# cap.release()
# out.release()
# print(f'\n✅ Done! Saved: {OUTPUT_PATH}')

In [ ]:
# from ultralytics import YOLO
# import cv2
# import numpy as np

# # ── Load YOLOv8 model ─────────────────────────────────────────────────────────
# print('🔄 Loading YOLOv8 model...')
# model = YOLO('yolov8n.pt')
# print('✅ Model loaded!')

# # ── Helper functions ──────────────────────────────────────────────────────────
# def side_of_line(p, a, b):
#     """Returns + or - value based on which side of line AB point P is on."""
#     return (b[0]-a[0])*(p[1]-a[1]) - (b[1]-a[1])*(p[0]-a[0])

# def get_loiter_zone_side(zone_points, tw_p1, tw_p2):
#     """
#     Returns the sign (+1 or -1) representing which side of the
#     tripwire line the loiter zone center is on.
#     """
#     if len(zone_points) < 3:
#         return None
#     cx = np.mean([p[0] for p in zone_points])
#     cy = np.mean([p[1] for p in zone_points])
#     val = side_of_line((cx, cy), tw_p1, tw_p2)
#     return 1 if val >= 0 else -1

# def check_tripwire_cross_from_zone_side(prev_center, curr_center, p1, p2, zone_side):
#     """
#     Returns True ONLY if person crossed the tripwire AND
#     was previously on the SAME side as the loiter zone (zone_side).
#     """
#     if prev_center is None or zone_side is None:
#         return False
#     prev_val = side_of_line(prev_center, p1, p2)
#     curr_val = side_of_line(curr_center, p1, p2)

#     # No crossing happened
#     if prev_val * curr_val >= 0:
#         return False

#     # Person was on zone side before crossing?
#     prev_side = 1 if prev_val >= 0 else -1
#     return prev_side == zone_side

# def is_on_zone_side(center, p1, p2, zone_side):
#     """Check if a point is currently on the loiter zone side of the tripwire."""
#     val = side_of_line(center, p1, p2)
#     curr_side = 1 if val >= 0 else -1
#     return curr_side == zone_side

# def is_near_tripwire(center, p1, p2, threshold=50):
#     """Perpendicular distance from point to tripwire line segment."""
#     x0,y0 = center
#     x1,y1 = p1
#     x2,y2 = p2
#     dx, dy = x2-x1, y2-y1
#     length = max(np.sqrt(dx*dx + dy*dy), 1)
#     dist = abs(dy*x0 - dx*y0 + x2*y1 - y2*x1) / length
#     t = ((x0-x1)*dx + (y0-y1)*dy) / (length*length)
#     return dist < threshold and 0 <= t <= 1

# def point_in_polygon(point, polygon):
#     if len(polygon) < 3:
#         return False
#     x, y = point
#     inside = False
#     px, py = polygon[-1]
#     for cx, cy in polygon:
#         if ((cy > y) != (py > y)) and (x < (px-cx)*(y-cy)/(py-cy)+cx):
#             inside = not inside
#         px, py = cx, cy
#     return inside

# def draw_alert_banner(frame, text, color_bgr, alpha=0.65):
#     """Draw a semi-transparent alert banner at the top of the frame."""
#     H, W = frame.shape[:2]
#     banner_h = 62
#     overlay = frame.copy()
#     cv2.rectangle(overlay, (0, 0), (W, banner_h), color_bgr, -1)
#     cv2.addWeighted(overlay, alpha, frame, 1-alpha, 0, frame)
#     font       = cv2.FONT_HERSHEY_DUPLEX
#     font_scale = 1.05
#     thickness  = 2
#     text_size  = cv2.getTextSize(text, font, font_scale, thickness)[0]
#     text_x = (W - text_size[0]) // 2
#     text_y = (banner_h + text_size[1]) // 2
#     cv2.putText(frame, text, (text_x, text_y), font, font_scale, (255,255,255), thickness)

# # ── Pre-compute which side of tripwire the loiter zone is on ──────────────────
# ZONE_SIDE = None
# if len(tripwire_points) == 2 and len(zone_points) >= 3:
#     ZONE_SIDE = get_loiter_zone_side(zone_points, tripwire_points[0], tripwire_points[1])
#     side_label = 'RIGHT' if ZONE_SIDE == 1 else 'LEFT'
#     print(f'📐 Loiter zone is on the [{side_label}] side of the tripwire.')
#     print(f'   ✅ Tripwire alert will ONLY trigger for persons crossing FROM that side.')
# else:
#     print('⚠️  Define both tripwire (2pts) and zone (3+pts) for directional alert logic.')

# # ── Video setup ───────────────────────────────────────────────────────────────
# cap   = cv2.VideoCapture(VIDEO_PATH)
# FPS   = cap.get(cv2.CAP_PROP_FPS) or 25
# TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
# W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# OUTPUT_PATH = 'output_detection.mp4'
# fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# out    = cv2.VideoWriter(OUTPUT_PATH, fourcc, FPS, (W, H))

# print(f'📹 Video: {W}x{H} @ {FPS:.1f} FPS | {TOTAL} frames')

# # ── Tracking state ─────────────────────────────────────────────────────────
# prev_centers             = {}
# zone_entry_time          = {}
# crossed_ids              = set()
# tripwire_alert_ids       = set()
# alert_log                = []
# LOITER_DISPLAY_INTERVAL  = 3
# last_loiter_display      = {}

# frame_idx = 0
# print('\n🚀 Running detection...\n')

# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     timestamp = frame_idx / FPS
#     frame_idx += 1

#     results = model.track(frame, persist=True, classes=[0],
#                           conf=CONF_THRESHOLD, verbose=False)
#     annotated = frame.copy()

#     # ── Draw tripwire ─────────────────────────────────────────────────────────
#     if len(tripwire_points) == 2:
#         tw_color = (0,0,255) if tripwire_alert_ids else (0,220,255)
#         cv2.line(annotated, tripwire_points[0], tripwire_points[1], tw_color, 3)
#         cv2.putText(annotated, 'TRIPWIRE',
#                     (tripwire_points[0][0], tripwire_points[0][1]-12),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.7, tw_color, 2)

#         # ── Draw side labels on video ──────────────────────────────────────
#         if ZONE_SIDE is not None:
#             mid_x = (tripwire_points[0][0] + tripwire_points[1][0]) // 2
#             mid_y = (tripwire_points[0][1] + tripwire_points[1][1]) // 2
#             # Offset labels perpendicular to line
#             dx = tripwire_points[1][0] - tripwire_points[0][0]
#             dy = tripwire_points[1][1] - tripwire_points[0][1]
#             length = max(np.sqrt(dx*dx+dy*dy), 1)
#             perp_x = int(-dy/length * 40)
#             perp_y = int(dx/length * 40)
#             # Zone side label
#             cv2.putText(annotated, '⚠ ALERT SIDE',
#                         (mid_x + perp_x * ZONE_SIDE, mid_y + perp_y * ZONE_SIDE),
#                         cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,0,255), 2)
#             # Safe side label
#             cv2.putText(annotated, 'SAFE SIDE',
#                         (mid_x - perp_x * ZONE_SIDE, mid_y - perp_y * ZONE_SIDE),
#                         cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,220,0), 2)

#     # ── Draw loiter zone ──────────────────────────────────────────────────────
#     if len(zone_points) >= 3:
#         pts_arr = np.array(zone_points, dtype=np.int32).reshape((-1,1,2))
#         overlay = annotated.copy()
#         cv2.fillPoly(overlay, [pts_arr], (255,100,0))
#         cv2.addWeighted(overlay, 0.2, annotated, 0.8, 0, annotated)
#         cv2.polylines(annotated, [pts_arr], True, (255,140,0), 2)
#         cx = int(np.mean([p[0] for p in zone_points]))
#         cy = int(np.mean([p[1] for p in zone_points]))
#         cv2.putText(annotated, 'LOITER ZONE', (cx-60, cy),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,140,0), 2)

#     # ── Per-frame alert flags ─────────────────────────────────────────────────
#     active_tripwire_alert         = False
#     active_loiter_alert           = False
#     active_loiter_ids             = []
#     tripwire_alert_ids_this_frame = set()

#     # ── Process detections ────────────────────────────────────────────────────
#     if results[0].boxes is not None and results[0].boxes.id is not None:
#         boxes     = results[0].boxes.xyxy.cpu().numpy().astype(int)
#         track_ids = results[0].boxes.id.cpu().numpy().astype(int)
#         confs     = results[0].boxes.conf.cpu().numpy()

#         for box, tid, conf in zip(boxes, track_ids, confs):
#             x1,y1,x2,y2 = box
#             cx_p = (x1+x2)//2
#             cy_p = (y1+y2)//2
#             center = (cx_p, cy_p)
#             color      = (0,220,0)
#             label_lines = [f'ID:{tid}  {conf:.2f}']

#             # ── Tripwire directional crossing check ───────────────────────
#             if len(tripwire_points) == 2:
#                 crossed_from_zone_side = check_tripwire_cross_from_zone_side(
#                     prev_centers.get(tid), center,
#                     tripwire_points[0], tripwire_points[1],
#                     ZONE_SIDE
#                 )

#                 if crossed_from_zone_side and tid not in crossed_ids:
#                     crossed_ids.add(tid)
#                     alert_log.append({
#                         'type'      : 'Tripwire',
#                         'track_id'  : int(tid),
#                         'timestamp' : f'{timestamp:.2f}s',
#                         'frame'     : frame_idx,
#                         'location'  : center
#                     })
#                     print(f'  🚨 TRIPWIRE (from zone side) | ID:{tid} | {timestamp:.2f}s')

#                 # Sustained alert: person near tripwire AND came from zone side
#                 if (is_near_tripwire(center, tripwire_points[0], tripwire_points[1], threshold=50)
#                         and tid in crossed_ids):
#                     tripwire_alert_ids_this_frame.add(tid)
#                     active_tripwire_alert = True
#                     color = (0,0,255)
#                     label_lines.append('⚡ TRIPWIRE!')

#                 # Also sustain alert if person is currently on the NON-zone side
#                 # (just crossed over) and still near the line
#                 elif (is_near_tripwire(center, tripwire_points[0], tripwire_points[1], threshold=50)
#                         and not is_on_zone_side(center, tripwire_points[0], tripwire_points[1], ZONE_SIDE)
#                         and tid in crossed_ids):
#                     tripwire_alert_ids_this_frame.add(tid)
#                     active_tripwire_alert = True
#                     color = (0,0,255)
#                     label_lines.append('⚡ TRIPWIRE!')

#             # ── Loitering check ───────────────────────────────────────────
#             if len(zone_points) >= 3 and point_in_polygon(center, zone_points):
#                 if tid not in zone_entry_time:
#                     zone_entry_time[tid] = timestamp
#                 dwell = timestamp - zone_entry_time[tid]

#                 if dwell >= LOITER_SECONDS:
#                     active_loiter_alert = True
#                     active_loiter_ids.append((tid, dwell))
#                     color = (0,100,255)
#                     label_lines.append(f'⏱ {dwell:.0f}s')

#                     last_log = next((a for a in reversed(alert_log)
#                                      if a['type']=='Loitering'
#                                      and a['track_id']==int(tid)), None)
#                     if (last_log is None or
#                             timestamp - float(last_log['timestamp'].replace('s','')) >= 2):
#                         alert_log.append({
#                             'type'      : 'Loitering',
#                             'track_id'  : int(tid),
#                             'timestamp' : f'{timestamp:.2f}s',
#                             'frame'     : frame_idx,
#                             'dwell_sec' : f'{dwell:.1f}'
#                         })
#                         print(f'  ⏱️  LOITERING | ID:{tid} | {dwell:.1f}s | {timestamp:.2f}s')
#             else:
#                 zone_entry_time.pop(tid, None)

#             # ── Draw box & labels ─────────────────────────────────────────
#             cv2.rectangle(annotated, (x1,y1), (x2,y2), color, 2)
#             cv2.circle(annotated, center, 5, color, -1)
#             for i, lbl in enumerate(label_lines):
#                 cv2.putText(annotated, lbl, (x1, y1 - 12 - i*18),
#                             cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

#             prev_centers[tid] = center

#     tripwire_alert_ids = tripwire_alert_ids_this_frame

#     # ── ON-SCREEN BANNERS ─────────────────────────────────────────────────────
#     if active_tripwire_alert:
#         draw_alert_banner(annotated,
#                           '🚨  ALERT: TRIPWIRE BREACH FROM RESTRICTED SIDE!',
#                           (0, 0, 180), alpha=0.65)
#     elif active_loiter_alert:
#         tid_max, dwell_max = max(active_loiter_ids, key=lambda x: x[1])
#         last_shown = last_loiter_display.get(tid_max, -999)
#         if timestamp - last_shown >= LOITER_DISPLAY_INTERVAL:
#             last_loiter_display[tid_max] = timestamp
#         if timestamp - last_loiter_display.get(tid_max, -999) < 1.5:
#             draw_alert_banner(annotated,
#                               f'⚠️  ALERT: LOITERING DETECTED  ({dwell_max:.0f}s)',
#                               (0, 80, 200), alpha=0.60)

#     # ── Timestamp ─────────────────────────────────────────────────────────────
#     cv2.putText(annotated, f'Frame:{frame_idx}  Time:{timestamp:.2f}s',
#                 (10, H-15), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)

#     out.write(annotated)

#     if frame_idx % 50 == 0:
#         pct = frame_idx/TOTAL*100 if TOTAL > 0 else 0
#         print(f'  Progress: {frame_idx}/{TOTAL} ({pct:.1f}%)')

# cap.release()
# out.release()
# print(f'\n✅ Done! Saved: {OUTPUT_PATH}')

In [ ]:
# from ultralytics import YOLO
# import cv2
# import numpy as np

# # ── Load model ────────────────────────────────────────────────────────────────
# print('🔄 Loading YOLOv8 model...')
# model = YOLO('yolov8n.pt')
# print('✅ Model loaded!')

# # ── YOLO class names ──────────────────────────────────────────────────────────
# COCO_CLASSES = model.names  # dict: {0: 'person', 1: 'bicycle', 2: 'car', ...}

# def get_label(cls_id, conf):
#     """Return class name if known, else 'Unknown Object'."""
#     if cls_id is not None and int(cls_id) in COCO_CLASSES:
#         return COCO_CLASSES[int(cls_id)].capitalize()
#     return 'Unknown Object'

# # ── Helper functions ──────────────────────────────────────────────────────────
# def side_of_line(p, a, b):
#     return (b[0]-a[0])*(p[1]-a[1]) - (b[1]-a[1])*(p[0]-a[0])

# def get_loiter_zone_side(zone_points, tw_p1, tw_p2):
#     if len(zone_points) < 3:
#         return None
#     cx = np.mean([p[0] for p in zone_points])
#     cy = np.mean([p[1] for p in zone_points])
#     val = side_of_line((cx, cy), tw_p1, tw_p2)
#     return 1 if val >= 0 else -1

# def check_tripwire_cross_from_zone_side(prev_center, curr_center, p1, p2, zone_side):
#     if prev_center is None or zone_side is None:
#         return False
#     prev_val = side_of_line(prev_center, p1, p2)
#     curr_val = side_of_line(curr_center, p1, p2)
#     if prev_val * curr_val >= 0:
#         return False
#     prev_side = 1 if prev_val >= 0 else -1
#     return prev_side == zone_side

# def is_on_zone_side(center, p1, p2, zone_side):
#     val = side_of_line(center, p1, p2)
#     return (1 if val >= 0 else -1) == zone_side

# def is_near_tripwire(center, p1, p2, threshold=50):
#     x0,y0 = center
#     x1,y1 = p1
#     x2,y2 = p2
#     dx, dy = x2-x1, y2-y1
#     length = max(np.sqrt(dx*dx + dy*dy), 1)
#     dist = abs(dy*x0 - dx*y0 + x2*y1 - y2*x1) / length
#     t = ((x0-x1)*dx + (y0-y1)*dy) / (length*length)
#     return dist < threshold and 0 <= t <= 1

# def point_in_polygon(point, polygon):
#     if len(polygon) < 3:
#         return False
#     x, y = point
#     inside = False
#     px, py = polygon[-1]
#     for cx, cy in polygon:
#         if ((cy > y) != (py > y)) and (x < (px-cx)*(y-cy)/(py-cy)+cx):
#             inside = not inside
#         px, py = cx, cy
#     return inside

# def draw_alert_banner(frame, text, color_bgr, alpha=0.65):
#     H, W = frame.shape[:2]
#     banner_h = 62
#     overlay = frame.copy()
#     cv2.rectangle(overlay, (0, 0), (W, banner_h), color_bgr, -1)
#     cv2.addWeighted(overlay, alpha, frame, 1-alpha, 0, frame)
#     font       = cv2.FONT_HERSHEY_DUPLEX
#     font_scale = 1.05
#     thickness  = 2
#     text_size  = cv2.getTextSize(text, font, font_scale, thickness)[0]
#     text_x = (W - text_size[0]) // 2
#     text_y = (banner_h + text_size[1]) // 2
#     cv2.putText(frame, text, (text_x, text_y), font, font_scale, (255,255,255), thickness)

# # ── Pre-compute loiter zone side ──────────────────────────────────────────────
# ZONE_SIDE = None
# if len(tripwire_points) == 2 and len(zone_points) >= 3:
#     ZONE_SIDE = get_loiter_zone_side(zone_points, tripwire_points[0], tripwire_points[1])
#     side_label = 'RIGHT' if ZONE_SIDE == 1 else 'LEFT'
#     print(f'📐 Loiter zone is on the [{side_label}] side of tripwire.')
#     print(f'   ✅ Tripwire alert fires ONLY for objects crossing FROM that side.')

# # ── Video setup ───────────────────────────────────────────────────────────────
# cap   = cv2.VideoCapture(VIDEO_PATH)
# FPS   = cap.get(cv2.CAP_PROP_FPS) or 25
# TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
# W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# OUTPUT_PATH = 'output_detection.mp4'
# fourcc = cv2.VideoWriter_fourcc(*'mp4v')
# out    = cv2.VideoWriter(OUTPUT_PATH, fourcc, FPS, (W, H))
# print(f'📹 Video: {W}x{H} @ {FPS:.1f} FPS | {TOTAL} frames')

# # ── Tracking state ────────────────────────────────────────────────────────────
# prev_centers            = {}
# zone_entry_time         = {}
# crossed_ids             = set()
# tripwire_alert_ids      = set()
# alert_log               = []
# last_loiter_display     = {}

# frame_idx = 0
# print('\n🚀 Running detection...\n')

# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     timestamp = frame_idx / FPS
#     frame_idx += 1

#     # ── YOLO inference — all classes ──────────────────────────────────────────
#     results = model.track(frame, persist=True,
#                           classes=DETECT_CLASSES,
#                           conf=CONF_THRESHOLD,
#                           verbose=False)

#     annotated = frame.copy()

#     # ── Draw tripwire ─────────────────────────────────────────────────────────
#     if len(tripwire_points) == 2:
#         tw_color = (0,0,255) if tripwire_alert_ids else (0,220,255)
#         cv2.line(annotated, tripwire_points[0], tripwire_points[1], tw_color, 3)
#         cv2.putText(annotated, 'TRIPWIRE',
#                     (tripwire_points[0][0], tripwire_points[0][1]-12),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.7, tw_color, 2)
#         if ZONE_SIDE is not None:
#             mid_x = (tripwire_points[0][0] + tripwire_points[1][0]) // 2
#             mid_y = (tripwire_points[0][1] + tripwire_points[1][1]) // 2
#             dx = tripwire_points[1][0] - tripwire_points[0][0]
#             dy = tripwire_points[1][1] - tripwire_points[0][1]
#             length = max(np.sqrt(dx*dx+dy*dy), 1)
#             perp_x = int(-dy/length * 40)
#             perp_y = int(dx/length * 40)
#             cv2.putText(annotated, '⚠ ALERT SIDE',
#                         (mid_x + perp_x*ZONE_SIDE, mid_y + perp_y*ZONE_SIDE),
#                         cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,0,255), 2)
#             cv2.putText(annotated, 'SAFE SIDE',
#                         (mid_x - perp_x*ZONE_SIDE, mid_y - perp_y*ZONE_SIDE),
#                         cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,220,0), 2)

#     # ── Draw loiter zone ──────────────────────────────────────────────────────
#     if len(zone_points) >= 3:
#         pts_arr = np.array(zone_points, dtype=np.int32).reshape((-1,1,2))
#         overlay = annotated.copy()
#         cv2.fillPoly(overlay, [pts_arr], (255,100,0))
#         cv2.addWeighted(overlay, 0.2, annotated, 0.8, 0, annotated)
#         cv2.polylines(annotated, [pts_arr], True, (255,140,0), 2)
#         cx = int(np.mean([p[0] for p in zone_points]))
#         cy = int(np.mean([p[1] for p in zone_points]))
#         cv2.putText(annotated, 'LOITER ZONE', (cx-60, cy),
#                     cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,140,0), 2)

#     # ── Per-frame flags ───────────────────────────────────────────────────────
#     active_tripwire_alert         = False
#     active_loiter_alert           = False
#     active_loiter_ids             = []
#     tripwire_alert_ids_this_frame = set()

#     # ── Process detections ────────────────────────────────────────────────────
#     if results[0].boxes is not None:
#         boxes  = results[0].boxes.xyxy.cpu().numpy().astype(int)
#         confs  = results[0].boxes.conf.cpu().numpy()
#         cls_ids = results[0].boxes.cls.cpu().numpy() if results[0].boxes.cls is not None else [None]*len(boxes)

#         # Track IDs (may be None if tracking lost)
#         if results[0].boxes.id is not None:
#             track_ids = results[0].boxes.id.cpu().numpy().astype(int)
#         else:
#             track_ids = list(range(len(boxes)))  # fallback IDs

#         for box, tid, conf, cls_id in zip(boxes, track_ids, confs, cls_ids):
#             x1,y1,x2,y2 = box
#             cx_p = (x1+x2)//2
#             cy_p = (y1+y2)//2
#             center = (cx_p, cy_p)

#             obj_label  = get_label(cls_id, conf)
#             color      = (0,220,0)
#             label_lines = [f'{obj_label}  ID:{tid}  {conf:.2f}']

#             # ── Tripwire check ────────────────────────────────────────────
#             if len(tripwire_points) == 2:
#                 crossed = check_tripwire_cross_from_zone_side(
#                     prev_centers.get(tid), center,
#                     tripwire_points[0], tripwire_points[1],
#                     ZONE_SIDE)

#                 if crossed and tid not in crossed_ids:
#                     crossed_ids.add(tid)
#                     alert_log.append({
#                         'type'      : 'Tripwire',
#                         'object'    : obj_label,
#                         'track_id'  : int(tid),
#                         'timestamp' : f'{timestamp:.2f}s',
#                         'frame'     : frame_idx,
#                         'location'  : center
#                     })
#                     print(f'  🚨 TRIPWIRE | {obj_label} | ID:{tid} | {timestamp:.2f}s')

#                 # Sustained alert while near line
#                 if is_near_tripwire(center, tripwire_points[0], tripwire_points[1], threshold=50) \
#                         and tid in crossed_ids:
#                     tripwire_alert_ids_this_frame.add(tid)
#                     active_tripwire_alert = True
#                     color = (0,0,255)
#                     label_lines.append('⚡ TRIPWIRE!')

#             # ── Loitering check ───────────────────────────────────────────
#             if len(zone_points) >= 3 and point_in_polygon(center, zone_points):
#                 if tid not in zone_entry_time:
#                     zone_entry_time[tid] = timestamp
#                 dwell = timestamp - zone_entry_time[tid]

#                 if dwell >= LOITER_SECONDS:
#                     active_loiter_alert = True
#                     active_loiter_ids.append((tid, dwell, obj_label))
#                     color = (0,100,255)
#                     label_lines.append(f'⏱ {dwell:.0f}s')

#                     last_log = next((a for a in reversed(alert_log)
#                                      if a['type']=='Loitering'
#                                      and a['track_id']==int(tid)), None)
#                     if last_log is None or \
#                             timestamp - float(last_log['timestamp'].replace('s','')) >= 2:
#                         alert_log.append({
#                             'type'      : 'Loitering',
#                             'object'    : obj_label,
#                             'track_id'  : int(tid),
#                             'timestamp' : f'{timestamp:.2f}s',
#                             'frame'     : frame_idx,
#                             'dwell_sec' : f'{dwell:.1f}'
#                         })
#                         print(f'  ⏱️  LOITERING | {obj_label} | ID:{tid} | {dwell:.1f}s | {timestamp:.2f}s')
#             else:
#                 zone_entry_time.pop(tid, None)

#             # ── Draw box ──────────────────────────────────────────────────
#             cv2.rectangle(annotated, (x1,y1), (x2,y2), color, 2)
#             cv2.circle(annotated, center, 5, color, -1)
#             for i, lbl in enumerate(label_lines):
#                 cv2.putText(annotated, lbl, (x1, y1 - 12 - i*18),
#                             cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

#             prev_centers[tid] = center

#     tripwire_alert_ids = tripwire_alert_ids_this_frame

#     # ── Alert banners ─────────────────────────────────────────────────────────
#     if active_tripwire_alert:
#         draw_alert_banner(annotated,
#                           '🚨  ALERT: TRIPWIRE BREACH DETECTED!',
#                           (0,0,180), alpha=0.65)
#     elif active_loiter_alert:
#         tid_max, dwell_max, lbl_max = max(active_loiter_ids, key=lambda x: x[1])
#         last_shown = last_loiter_display.get(tid_max, -999)
#         if timestamp - last_shown >= LOITER_DISPLAY_INTERVAL:
#             last_loiter_display[tid_max] = timestamp
#         if timestamp - last_loiter_display.get(tid_max, -999) < 1.5:
#             draw_alert_banner(annotated,
#                               f'⚠️  LOITERING: {lbl_max}  ({dwell_max:.0f}s)',
#                               (0,80,200), alpha=0.60)

#     # ── Timestamp ─────────────────────────────────────────────────────────────
#     cv2.putText(annotated, f'Frame:{frame_idx}  Time:{timestamp:.2f}s',
#                 (10, H-15), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)

#     out.write(annotated)

#     if frame_idx % 50 == 0:
#         pct = frame_idx/TOTAL*100 if TOTAL > 0 else 0
#         print(f'  Progress: {frame_idx}/{TOTAL} ({pct:.1f}%)')

# cap.release()
# out.release()
# print(f'\n✅ Done! Saved: {OUTPUT_PATH}')

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np

# ── Load model ────────────────────────────────────────────────────────────────
print('🔄 Loading YOLOv8 model...')
model = YOLO('yolov8n.pt')
print('✅ Model loaded!')

# ── YOLO class names ──────────────────────────────────────────────────────────
COCO_CLASSES = model.names

def get_label(cls_id, conf):
    if cls_id is not None and int(cls_id) in COCO_CLASSES:
        return COCO_CLASSES[int(cls_id)].capitalize()
    return 'Unknown Object'

# ── Helper functions ──────────────────────────────────────────────────────────
def side_of_line(p, a, b):
    return (b[0]-a[0])*(p[1]-a[1]) - (b[1]-a[1])*(p[0]-a[0])

def get_loiter_zone_side(zone_points, tw_p1, tw_p2):
    if len(zone_points) < 4:
        return None
    cx = np.mean([p[0] for p in zone_points])
    cy = np.mean([p[1] for p in zone_points])
    val = side_of_line((cx, cy), tw_p1, tw_p2)
    return 1 if val >= 0 else -1

def check_tripwire_cross_from_zone_side(prev_center, curr_center, p1, p2, zone_side):
    if prev_center is None or zone_side is None:
        return False
    prev_val = side_of_line(prev_center, p1, p2)
    curr_val = side_of_line(curr_center, p1, p2)
    if prev_val * curr_val >= 0:
        return False
    prev_side = 1 if prev_val >= 0 else -1
    return prev_side == zone_side

def is_on_zone_side(center, p1, p2, zone_side):
    val = side_of_line(center, p1, p2)
    return (1 if val >= 0 else -1) == zone_side

def is_near_tripwire(center, p1, p2, threshold=50):
    x0,y0 = center
    x1,y1 = p1
    x2,y2 = p2
    dx, dy = x2-x1, y2-y1
    length = max(np.sqrt(dx*dx + dy*dy), 1)
    dist = abs(dy*x0 - dx*y0 + x2*y1 - y2*x1) / length
    t = ((x0-x1)*dx + (y0-y1)*dy) / (length*length)
    return dist < threshold and 0 <= t <= 1

def point_in_polygon(point, polygon):
    if len(polygon) < 4:
        return False
    x, y = point
    inside = False
    px, py = polygon[-1]
    for cx, cy in polygon:
        if ((cy > y) != (py > y)) and (x < (px-cx)*(y-cy)/(py-cy)+cx):
            inside = not inside
        px, py = cx, cy
    return inside

def draw_alert_banner(frame, text, color_bgr, alpha=0.65):
    H, W = frame.shape[:2]
    banner_h = 62
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (W, banner_h), color_bgr, -1)
    cv2.addWeighted(overlay, alpha, frame, 1-alpha, 0, frame)
    font       = cv2.FONT_HERSHEY_DUPLEX
    font_scale = 1.05
    thickness  = 2
    text_size  = cv2.getTextSize(text, font, font_scale, thickness)[0]
    text_x = (W - text_size[0]) // 2
    text_y = (banner_h + text_size[1]) // 2
    cv2.putText(frame, text, (text_x, text_y), font, font_scale, (255,255,255), thickness)

# ── Pre-compute loiter zone side ──────────────────────────────────────────────
ZONE_SIDE = None
if len(tripwire_points) == 2 and len(zone_points) >= 4:
    ZONE_SIDE = get_loiter_zone_side(zone_points, tripwire_points[0], tripwire_points[1])
    side_label = 'RIGHT' if ZONE_SIDE == 1 else 'LEFT'
    print(f'📐 Loiter zone is on the [{side_label}] side of tripwire.')
    print(f'   ✅ Tripwire alert fires ONLY for objects crossing FROM that side.')
elif len(zone_points) > 0 and len(zone_points) < 4:
    print(f'⚠️  Zone has only {len(zone_points)} points — minimum 4 required. Zone detection disabled.')
else:
    print('ℹ️  No zone defined — loitering detection disabled.')

# ── Video setup ───────────────────────────────────────────────────────────────
cap   = cv2.VideoCapture(VIDEO_PATH)
FPS   = cap.get(cv2.CAP_PROP_FPS) or 25
TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

OUTPUT_PATH = 'output_detection.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(OUTPUT_PATH, fourcc, FPS, (W, H))
print(f'📹 Video: {W}x{H} @ {FPS:.1f} FPS | {TOTAL} frames')

# ── Tracking state ────────────────────────────────────────────────────────────
prev_centers            = {}
zone_entry_time         = {}
crossed_ids             = set()
tripwire_alert_ids      = set()
alert_log               = []
last_loiter_display     = {}

frame_idx = 0
print('\n🚀 Running detection...\n')

while True:
    ret, frame = cap.read()
    if not ret:
        break

    timestamp = frame_idx / FPS
    frame_idx += 1

    # ── YOLO inference ────────────────────────────────────────────────────────
    results = model.track(frame, persist=True,
                          classes=DETECT_CLASSES,
                          conf=CONF_THRESHOLD,
                          verbose=False)

    annotated = frame.copy()

    # ── Draw tripwire ─────────────────────────────────────────────────────────
    if len(tripwire_points) == 2:
        tw_color = (0,0,255) if tripwire_alert_ids else (0,220,255)
        cv2.line(annotated, tripwire_points[0], tripwire_points[1], tw_color, 3)
        cv2.putText(annotated, 'TRIPWIRE',
                    (tripwire_points[0][0], tripwire_points[0][1]-12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, tw_color, 2)
        if ZONE_SIDE is not None:
            mid_x = (tripwire_points[0][0] + tripwire_points[1][0]) // 2
            mid_y = (tripwire_points[0][1] + tripwire_points[1][1]) // 2
            dx = tripwire_points[1][0] - tripwire_points[0][0]
            dy = tripwire_points[1][1] - tripwire_points[0][1]
            length = max(np.sqrt(dx*dx+dy*dy), 1)
            perp_x = int(-dy/length * 40)
            perp_y = int(dx/length * 40)
            cv2.putText(annotated, '⚠ ALERT SIDE',
                        (mid_x + perp_x*ZONE_SIDE, mid_y + perp_y*ZONE_SIDE),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,0,255), 2)
            cv2.putText(annotated, 'SAFE SIDE',
                        (mid_x - perp_x*ZONE_SIDE, mid_y - perp_y*ZONE_SIDE),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,220,0), 2)

    # ── Draw loiter zone ──────────────────────────────────────────────────────
    if len(zone_points) >= 4:
        pts_arr = np.array(zone_points, dtype=np.int32).reshape((-1,1,2))
        overlay = annotated.copy()
        cv2.fillPoly(overlay, [pts_arr], (255,100,0))
        cv2.addWeighted(overlay, 0.2, annotated, 0.8, 0, annotated)
        cv2.polylines(annotated, [pts_arr], True, (255,140,0), 2)
        cx = int(np.mean([p[0] for p in zone_points]))
        cy = int(np.mean([p[1] for p in zone_points]))
        cv2.putText(annotated, 'LOITER ZONE', (cx-60, cy),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,140,0), 2)

    # ── Per-frame flags ───────────────────────────────────────────────────────
    active_tripwire_alert         = False
    active_loiter_alert           = False
    active_loiter_ids             = []
    tripwire_alert_ids_this_frame = set()

    # ── Process detections ────────────────────────────────────────────────────
    if results[0].boxes is not None:
        boxes   = results[0].boxes.xyxy.cpu().numpy().astype(int)
        confs   = results[0].boxes.conf.cpu().numpy()
        cls_ids = results[0].boxes.cls.cpu().numpy() \
                  if results[0].boxes.cls is not None else [None]*len(boxes)

        if results[0].boxes.id is not None:
            track_ids = results[0].boxes.id.cpu().numpy().astype(int)
        else:
            track_ids = list(range(len(boxes)))

        for box, tid, conf, cls_id in zip(boxes, track_ids, confs, cls_ids):
            x1,y1,x2,y2 = box
            cx_p = (x1+x2)//2
            cy_p = (y1+y2)//2
            center = (cx_p, cy_p)

            obj_label   = get_label(cls_id, conf)
            color       = (0,220,0)
            label_lines = [f'{obj_label}  ID:{tid}  {conf:.2f}']

            # ── Tripwire check ────────────────────────────────────────────
            if len(tripwire_points) == 2:
                crossed = check_tripwire_cross_from_zone_side(
                    prev_centers.get(tid), center,
                    tripwire_points[0], tripwire_points[1],
                    ZONE_SIDE)

                if crossed and tid not in crossed_ids:
                    crossed_ids.add(tid)
                    alert_log.append({
                        'type'      : 'Tripwire',
                        'object'    : obj_label,
                        'track_id'  : int(tid),
                        'timestamp' : f'{timestamp:.2f}s',
                        'frame'     : frame_idx,
                        'location'  : center
                    })
                    print(f'  🚨 TRIPWIRE | {obj_label} | ID:{tid} | {timestamp:.2f}s')

                if is_near_tripwire(center, tripwire_points[0], tripwire_points[1], threshold=50) \
                        and tid in crossed_ids:
                    tripwire_alert_ids_this_frame.add(tid)
                    active_tripwire_alert = True
                    color = (0,0,255)
                    label_lines.append('⚡ TRIPWIRE!')

            # ── Loitering check ───────────────────────────────────────────
            if len(zone_points) >= 4 and point_in_polygon(center, zone_points):
                if tid not in zone_entry_time:
                    zone_entry_time[tid] = timestamp
                dwell = timestamp - zone_entry_time[tid]

                if dwell >= LOITER_SECONDS:
                    active_loiter_alert = True
                    active_loiter_ids.append((tid, dwell, obj_label))
                    color = (0,100,255)
                    label_lines.append(f'⏱ {dwell:.0f}s')

                    last_log = next((a for a in reversed(alert_log)
                                     if a['type']=='Loitering'
                                     and a['track_id']==int(tid)), None)
                    if last_log is None or \
                            timestamp - float(last_log['timestamp'].replace('s','')) >= 2:
                        alert_log.append({
                            'type'      : 'Loitering',
                            'object'    : obj_label,
                            'track_id'  : int(tid),
                            'timestamp' : f'{timestamp:.2f}s',
                            'frame'     : frame_idx,
                            'dwell_sec' : f'{dwell:.1f}'
                        })
                        print(f'  ⏱️  LOITERING | {obj_label} | ID:{tid} | {dwell:.1f}s | {timestamp:.2f}s')
            else:
                zone_entry_time.pop(tid, None)

            # ── Draw box ──────────────────────────────────────────────────
            cv2.rectangle(annotated, (x1,y1), (x2,y2), color, 2)
            cv2.circle(annotated, center, 5, color, -1)
            for i, lbl in enumerate(label_lines):
                cv2.putText(annotated, lbl, (x1, y1 - 12 - i*18),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

            prev_centers[tid] = center

    tripwire_alert_ids = tripwire_alert_ids_this_frame

    # ── Alert banners ─────────────────────────────────────────────────────────
    if active_tripwire_alert:
        draw_alert_banner(annotated,
                          '🚨  ALERT: TRIPWIRE BREACH DETECTED!',
                          (0,0,180), alpha=0.65)
    elif active_loiter_alert:
        tid_max, dwell_max, lbl_max = max(active_loiter_ids, key=lambda x: x[1])
        last_shown = last_loiter_display.get(tid_max, -999)
        if timestamp - last_shown >= LOITER_DISPLAY_INTERVAL:
            last_loiter_display[tid_max] = timestamp
        if timestamp - last_loiter_display.get(tid_max, -999) < 1.5:
            draw_alert_banner(annotated,
                              f'⚠️  LOITERING: {lbl_max}  ({dwell_max:.0f}s)',
                              (0,80,200), alpha=0.60)

    # ── Timestamp ─────────────────────────────────────────────────────────────
    cv2.putText(annotated, f'Frame:{frame_idx}  Time:{timestamp:.2f}s',
                (10, H-15), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)

    out.write(annotated)

    if frame_idx % 50 == 0:
        pct = frame_idx/TOTAL*100 if TOTAL > 0 else 0
        print(f'  Progress: {frame_idx}/{TOTAL} ({pct:.1f}%)')

cap.release()
out.release()
print(f'\n✅ Done! Saved: {OUTPUT_PATH}')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🔄 Loading YOLOv8 model...
✅ Model loaded!
📐 Loiter zone is on the [RIGHT] side of tripwire.
   ✅ Tripwire alert fires ONLY for objects crossing FROM that side.
📹 Video: 1280x720 @ 30.0 FPS | 2631 frames

🚀 Running detection...

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 150ms
Prepared 1 package in 45ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

  Progress: 50/2631 (1.9%)
  Progress: 100/2631 (3.8%)
  Progress: 150/2631 (5.7%)
  ⏱️  LOITERING | Car

In [ ]:
from google.colab import files
import os

if os.path.exists('output_detection.mp4'):
    files.download('output_detection.mp4')
    print('✅ output_detection.mp4 downloaded')

# if os.path.exists('alert_log.csv'):
#     files.download('alert_log.csv')
#     print('✅ alert_log.csv downloaded')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ output_detection.mp4 downloaded
